In [ ]:
import urllib
print("Sua SENHA (IP) é:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

In [ ]:
!pip install streamlit #h2o -q
!npm install -g localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib
import os

st.set_page_config(page_title="Saúde - Risco Doença", layout="wide")

# Mensagem de aviso educacional
st.info("""
**Este conteúdo é destinado apenas para fins educacionais.** Os dados exibidos são ilustrativos e podem não corresponder a situações reais.
""")

@st.cache_resource
def load_model():
    model_path = "modelo_final_saude.pkl"
    if os.path.exists(model_path):
        return joblib.load(model_path)
    return None

model = load_model()

st.title("🩺 Risco Doença")
st.markdown("Preencha os dados abaixo para análise via Random Forest.")

if model is not None:
    try:
        # Recupera as colunas que o modelo espera
        expected_features = model.feature_names_in_.tolist()
    except AttributeError:
        st.error("O modelo não contém os nomes das colunas. Verifique o salvamento.")
        st.stop()

    with st.form("form_clinico"):
        col1, col2 = st.columns(2)
        user_data = {}
        
        for i, feature in enumerate(expected_features):
            with col1 if i % 2 == 0 else col2:                
                user_data[feature] = st.number_input(f"Valor para {feature}", value=0.0)
        
        enviar = st.form_submit_button("Realizar Predição")

    if enviar:
        # Garante a ordem correta das colunas
        input_df = pd.DataFrame([user_data])[expected_features]
        try:
            prediction = model.predict(input_df)
            probability = model.predict_proba(input_df)
            
            st.divider()
            if prediction[0] == 1:
                st.error(f"### ⚠️ ALTO RISCO DETECTADO")
                st.write(f"Probabilidade: {probability[0][1]:.2%}")
            else:
                st.success(f"### ✅ BAIXO RISCO DETECTADO")
                st.write(f"Probabilidade: {probability[0][0]:.2%}")
        
        except Exception as e:
            st.error(f"Erro no processamento: {e}")
            st.info("Verifique se todos os campos foram preenchidos corretamente.")

else:
    st.error("Arquivo 'modelo_final_saude.pkl' não encontrado!")
    st.info("Certifique-se de carregar o arquivo na pasta lateral do Colab.")

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501